# Module 3: Operating and Optimizing Apache Iceberg

This notebook contains code examples for Module 3 videos.

## Setup

Initialize Spark session for Module 3 examples.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module 3 - Operating and Optimizing Apache Iceberg") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Video 1: Write Operations and Optimization Strategies

In [ ]:
# Insert operation - adds new rows to the table
# This is purely additive and the fastest write operation

spark.sql("""
    INSERT INTO polaris.taxi.trips_by_day
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-09.parquet`
    WHERE tpep_pickup_datetime >= '2023-09-01'
      AND tpep_pickup_datetime < '2023-09-02'
""")

In [ ]:
# Delete operation with partition filter
# When aligned with partition boundaries, this is a metadata-only operation

spark.sql("""
    DELETE FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
""")

In [ ]:
# Overwrite operation - atomic delete + insert
# Replaces data matching a predicate with new data

spark.sql("""
    INSERT OVERWRITE polaris.taxi.trips_by_day
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
""")

In [ ]:
# Copy-on-Write (COW) mode update
# Rewrites entire affected files with modifications applied
# Slower writes, faster reads

# Set write mode to copy-on-write
spark.conf.set("spark.sql.iceberg.update.mode", "copy-on-write")

spark.sql("""
    UPDATE polaris.taxi.trips_by_day
    SET fare_amount = fare_amount * 1.1
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
      AND PULocationID = 132
""")

In [ ]:
# View the result of COW update
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        PULocationID,
        fare_amount
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
      AND PULocationID = 132
    LIMIT 10
""").show()

In [ ]:
# Merge-on-Read (MOR) mode update
# Writes delete files and small data files with changes
# Faster writes, slower reads

# Set write mode to merge-on-read
spark.conf.set("spark.sql.iceberg.update.mode", "merge-on-read")

spark.sql("""
    UPDATE polaris.taxi.trips_by_day
    SET tip_amount = tip_amount * 1.15
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
      AND PULocationID = 138
""")

In [ ]:
# View the result of MOR update
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        PULocationID,
        tip_amount
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-06-01'
      AND tpep_pickup_datetime < '2023-06-02'
      AND PULocationID = 138
    LIMIT 10
""").show()

In [ ]:
# View delete files in metadata
# MOR creates delete files that track which rows to ignore

spark.sql("""
    SELECT 
        file_path,
        record_count,
        file_size_in_bytes
    FROM polaris.taxi.trips_by_day.delete_files
    LIMIT 10
""").show(truncate=False)

## Video 3: Table Maintenance for Iceberg

In [ ]:
# Expire old snapshots to prevent metadata.json bloat
# This removes snapshots older than the retention period and enables garbage collection

spark.sql("""
    CALL polaris.system.expire_snapshots(
        table => 'polaris.taxi.trips_by_day',
        older_than => TIMESTAMP '2023-09-01 00:00:00',
        retain_last => 5
    )
""")

In [ ]:
# View remaining snapshots after expiration
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_by_day.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

In [ ]:
# Rewrite manifests to consolidate small manifest files
# This speeds up query planning by reducing metadata overhead

spark.sql("""
    CALL polaris.system.rewrite_manifests(
        table => 'polaris.taxi.trips_by_day'
    )
""")

In [ ]:
# Rewrite data files to compact small files into optimally-sized files
# This is the most important regular maintenance operation

spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'polaris.taxi.trips_by_day',
        strategy => 'binpack',
        options => map('target-file-size-bytes', '134217728')
    )
""")

In [ ]:
# Check file statistics after compaction
spark.sql("""
    SELECT 
        COUNT(*) as file_count,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb,
        ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_file_size_mb,
        ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_file_size_mb
    FROM polaris.taxi.trips_by_day.files
    WHERE content = 'DATA'
""").show()

In [ ]:
# Remove orphan files - only run when necessary
# This is expensive and should be used sparingly

spark.sql("""
    CALL polaris.system.remove_orphan_files(
        table => 'polaris.taxi.trips_by_day',
        older_than => TIMESTAMP '2023-09-01 00:00:00'
    )
""")

## Video 5: Sort Orders

In [ ]:
# Define a sort order on a table
# Sort orders organize data within files according to specified columns

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_sorted_example
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.taxi.trips_sorted_example
    WRITE ORDERED BY PULocationID, tpep_pickup_datetime
""")

In [ ]:
# Insert data into the sorted table
# Data will be sorted by PULocationID and timestamp before writing

spark.sql("""
    INSERT INTO polaris.taxi.trips_sorted_example
    SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

In [ ]:
# View min/max bounds for sorted columns
# Sort order creates better min/max statistics for file skipping

spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        readable_metrics.PULocationID.lower_bound as PULocationID_min,
        readable_metrics.PULocationID.upper_bound as PULocationID_max
    FROM polaris.taxi.trips_sorted_example.files
    ORDER BY partition, PULocationID_min
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Example: Partition by month (low cardinality)
# Fewer partitions = larger files, fewer to open

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month_example
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# View partition statistics for monthly partitioning
spark.sql("""
    SELECT 
        data_file.partition,
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(AVG(data_file.file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_month_example.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
""").show()

In [ ]:
# Example: Partition by day (high cardinality)
# More partitions = more files, smaller file sizes

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day_example
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# View partition statistics for daily partitioning
spark.sql("""
    SELECT 
        data_file.partition,
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(AVG(data_file.file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_day_example.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
    LIMIT 10
""").show()

In [ ]:
# Compare overall file statistics
spark.sql("""
    SELECT 
        'Monthly' as partition_type,
        COUNT(*) as total_files,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_month_example.files
    WHERE content = 'DATA'
    UNION ALL
    SELECT 
        'Daily' as partition_type,
        COUNT(*) as total_files,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_file_size_mb
    FROM polaris.taxi.trips_by_day_example.files
    WHERE content = 'DATA'
""").show()